# Atividade: Plano de Testes e Report de Bugs
## Sistema: Banco do Stark

**Analista de QA:** Vinicius Patrocinio Leite
**Objetivo:** Validar as regras de negócio bancárias e identificar vulnerabilidades no código inicial.

In [1]:
# Implementação do Sistema para Testes
class Conta:
    def __init__(self, titular: str):
        self.titular = titular
        self.saldo = 0.0
        self.extrato = []

class BancoDoStark:
    def __init__(self):
        self.contas = {}

    def criar_conta(self, titular: str):
        if titular in self.contas:
            print(f'ERRO - Conta já existe: {titular}')
            return
        self.contas[titular] = Conta(titular)
        print(f'OK - Conta criada para: {titular}')

    def depositar(self, titular: str, valor: float):
        if titular not in self.contas:
            print(f'ERRO - Conta não encontrada: {titular}')
            return
        self.contas[titular].saldo += valor
        self.contas[titular].extrato.append(('deposito', valor))
        print(f'OK - Depósito realizado: {valor}')

    def sacar(self, titular: str, valor: float):
        if titular not in self.contas:
            print(f'ERRO - Conta não encontrada: {titular}')
            return
        if self.contas[titular].saldo < valor:
            print('ERRO - Saldo insuficiente')
            return
        self.contas[titular].saldo -= valor
        self.contas[titular].extrato.append(('saque', valor))
        print(f'OK - Saque realizado: {valor}')

    def transferir(self, origem: str, destino: str, valor: float):
        if origem not in self.contas or destino not in self.contas:
            print('ERRO - Conta de origem ou destino inexistente')
            return
        if self.contas[origem].saldo < valor:
            print('ERRO - Saldo insuficiente para transferência')
            return
        self.contas[origem].saldo -= valor
        self.contas[destino].saldo += valor
        self.contas[origem].extrato.append(('transferencia_saida', valor))
        self.contas[destino].extrato.append(('transferencia_entrada', valor))
        print(f'OK - Transferência realizada: {valor}')

banco = BancoDoStark()
print('Ambiente de Testes Configurado.')

Ambiente de Testes Configurado.


## 1. Casos de Teste (Test Cases)

### **CT-001: Validar Depósito Nominal com Valor Positivo**
- **Objetivo:** Confirmar se o sistema credita valores válidos corretamente.
- **Pré-condição:** Conta 'tony' criada.
- **Passos:** Chamar `banco.depositar('tony', 1000)`.
- **Resultado Esperado:** Mensagem de sucesso e incremento no saldo.
- **Resultado Obtido:** OK - Depósito realizado: 1000.
- **Status:** **Pass**

---

### **CT-002: Impedir Transferência para Destinatário Inexistente**
- **Objetivo:** Garantir que fundos não sejam transferidos para contas fora da base.
- **Pré-condição:** Conta de origem com saldo suficiente.
- **Passos:** Tentar `banco.transferir('tony', 'thor', 50)`.
- **Resultado Esperado:** Mensagem de erro informando conta inexistente.
- **Resultado Obtido:** ERRO - Conta de origem ou destino inexistente.
- **Status:** **Pass**

---

### **CT-003: Criar Nova Conta de Usuário**
- **Objetivo:** Verificar a persistência de novos cadastros.
- **Passos:** Executar `banco.criar_conta('pepper')`.
- **Resultado Esperado:** Sistema deve retornar confirmação de criação.
- **Resultado Obtido:** OK - Conta criada para: pepper.
- **Status:** **Pass**

## 2. Bug Reports (Relatórios de Falhas)

### **BUG-001: Aceite de Saques com Valores Negativos**
- **Severidade:** Alta | **Prioridade:** Urgente
- **Descrição:** O sistema permite saques com valores menores que zero, o que acaba aumentando o saldo do usuário indevidamente (comportando-se como depósito).
- **Passos para Reproduzir:** `banco.sacar('pepper', -700)`.
- **Resultado Esperado:** Bloqueio da transação e erro de "Valor Inválido".
- **Resultado Obtido:** Saque processado com sucesso.
- **Impacto:** Risco crítico de fraude e corrupção de dados financeiros.

### **BUG-002: Falha na Validação de Titular Vazio**
- **Severidade:** Média | **Prioridade:** Média
- **Descrição:** O sistema permite a criação de contas onde o nome do titular é composto apenas por espaços em branco.
- **Passos para Reproduzir:** `banco.criar_conta('  ')`.
- **Resultado Esperado:** Erro de validação: "Nome do titular obrigatório".
- **Resultado Obtido:** Conta criada sem nome visível.
- **Impacto:** Inconsistência na base de dados e impossibilidade de identificação do cliente.

In [ ]:
# Execução de Evidências para o Report
print('Evidência BUG-001 (Saque Negativo):')
banco.criar_conta('pepper')
banco.sacar('pepper', -700)

print('\nEvidência BUG-002 (Titular Inválido):')
banco.criar_conta('  ')